In [1]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.ligsaf_utilities import create_rcf_utilities_system

In [2]:
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


In [3]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [4]:

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()
BT, WWT, gas_mixer = create_rcf_utilities_system()

rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: PUMP101> no pump type available at current power (1.96e+03 hp), head (3.23e+03 ft), kinematic viscosity (6.01e-07 m2/s), and NPSH (3.96 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:409: CostWarning: <SolvolysisReactor: RCF103_S> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: PUMP108> power (0 hp) i

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   1.86e-10 kmol/hr (0.012%)
- temperature 1.84e-03 K (0.00059%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  889
                    O2  219
[2] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  33.4
                    NaOH   15.1
[3] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin    156
                    Glucan    238
                    Xylan     84.5
                    Arabinan  1.26
                    Mannan    19
                    Galactan  7.2
[4] Meoh_in  
    phase: 'l

In [5]:
chems

CompiledChemicals([Water, Ethanol, AceticAcid, Furfural, Glycerol, H2SO4, NH3, LacticAcid, SuccinicAcid, P4O10, Lime, HNO3, NH4OH, Denaturant, DAP, AmmoniumAcetate, AmmoniumSulfate, NaNO3, Oil, HMF, N2, O2, CH4, H2S, SO2, CO2, NO2, NO, CO, Glucose, Xylose, Sucrose, CaSO4, Mannose, Galactose, Arabinose, CellulaseNutrients, Extract, Acetate, Tar, Ash, NaOH, Lignin, SolubleLignin, GlucoseOligomer, GalactoseOligomer, MannoseOligomer, XyloseOligomer, ArabinoseOligomer, Z_mobilis, T_reesei, Biomass, Cellulose, Protein, Enzyme, Glucan, Xylan, Xylitol, Cellobiose, CSL, DenaturedEnzyme, Arabinan, Mannan, Galactan, WWTsludge, Cellulase, Methanol, Hydrogen, Methane, Hexane, EthylAcetate, propylcyclohexane, Ethylene, Butene, Hex-1-ene, Dec-1-ene, Octadec-1-ene, Butane, Decane, Octadecane, Syndol, Nickel_SiAl, CobaltMolybdenum, Coal, NiC, ActivatedCarbon, Propylguaiacol, Propylsyringol, Syringaresinol, G_Dimer, S_Oligomer, G_Oligomer])


In [6]:
# Different flows for the two monomers - strange, probably have to do with the scaling function used and the monomer yields. 
# This is why constantly checking the system is crucial

In [7]:
    # Hydrodeoxygenation reactions

hydrodeoxygenation_rxn = bst.ParallelReaction([
    bst.Reaction('Propylguaiacol,l + 6Hydrogen,g -> 1propylcyclohexane,l + 2Water,l + Methane,g', reactant='Propylguaiacol', phases='lg',
                    X=1.0, basis='mol'),
    bst.Reaction('1Propylsyringol,l + 8Hydrogen,g -> 1propylcyclohexane,l + 3Water,l + 2Methane,g', reactant='Propylsyringol', phases='lg',
                    X=1.0, basis='mol'),
])

In [8]:
h2_in = bst.Stream(ID = 'Hydrogen_In', Hydrogen = 300 , units = 'kmol/hr', P = 5e6, phase = 'g')


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: Hydrogen_In> has been replaced in registry
  self._register(ID)


In [9]:
mixer = bst.units.Mixer(ins = (h2_in, F.RCF_Monomers), rigorous = True)
mixer.simulate()

In [10]:
#hydrodeoxygenation_rxn(mixer-0)

In [11]:
mixer-0

MultiStream: s44 from <Mixer: M3>
phases: ('g', 'l'), T: 373.03 K, P: 101325 Pa
flow (kmol/hr): (g) Hydrogen        300
                (l) Propylguaiacol  17.4
                    Propylsyringol  14.7


In [12]:
compressor = bst.units.IsentropicCompressor(ins = mixer-0, P = 5e6, vle = True)
compressor.simulate()

In [13]:
F.RCF_Monomers

Stream: RCF_Monomers from <Flash: FLASH301> to <Mixer: M3>
phase: 'l', T: 400 K, P: 101325 Pa
flow (kmol/hr): Propylguaiacol  17.4
                Propylsyringol  14.7


In [14]:
import biosteam as bst, numpy as np
from math import ceil

from typing import Optional
from biosteam.units.design_tools import (
    PressureVessel, 
)
 

class HydrodeoxygenationReactor(bst.Unit, bst.units.design_tools.PressureVessel):

    """
    Batch reactor for HDO of lignin oil
    Can take in complete RCF Oil, hence no need to isolate monomers 
    
    
    References
    ----------------------------------------------------------------------------------
        [1] Bruno Pandalone., et a;.
        "Optimum Lignin Oil - Finding the Most Suitable Feedstock to Replace Cycloalkanes in Sustainable Aviation Fuel (SAF)"
        ChemSusChem. 2025. 18(11). https://doi.org/10.1002/cssc.202402531
   -----------------------------------------------------------------------------------

    
    """



    _F_BM_default = {'Horizontal pressure vessel': 3.05,
                     'Vertical pressure vessel': 4.16,
                     'Platform and ladders': 1.}                   

    _N_ins = 1
    _N_outs = 1
    
    _units = {**PressureVessel._units,
              'Batch time': 'hr',
              'Turnaround time': 'hr',
              'Time on stream': 'hr',
              'Total beds': "",
              'Beds in service': "",
              'Total volume': 'm3',
              'Reactor volume': 'm3'}
    




    # Default operating temperature [K]
    T_default: float = 573.15                   # 300 C from [1]

    #: Default operating pressure [Pa]
    P_default:  float = 5e6                     # 5 MPa from [1]
    
    #: Default reaction time [hr]
    tau_default: float = 2                      # Total 2 hr reaction time, but could be higher (5 hr and 22 hr tested in [1])

    #: Default cleaning and unloading time (hr).
    tau_0_default: float  = 1                    # Assumed

    # Fixed bed configuration
    N_total_default: int =  3                      # Total beds (2 operating + 1 cleaning)

    N_working_default: int = 2                     # Beds operating at any time


    # Default catalyst bulk density [kg/m³]
    catalyst_bulk_density: float = 485            # Bulk density of the catalyst particle

    # Default free-space fraction of reactor volume
    free_frac_default: float = 0.10                # 10% kept free for gas disengagement / headspace
    
    # Default maximum vessel volume [m3]
    V_max_default: float = 600                     # Assumed

    # Aspect ratio (L/D of the reactor)
    aspect_ratio: float = 5.0                      # Assumed



    def _init(
            self,
            T: Optional[float] = None,
            P: Optional[float] = None,
            tau: Optional[float] = None,
            vessel_material: Optional[str] = None,
            vessel_type: Optional[str] = None,
            tau_0: Optional[float] = None,
            catalyst_bulk_density: Optional[float] = None,
            free_frac: Optional[float] = None,
            V_max: Optional[float] = None,
            V_max_limit: Optional[float] = None,
            aspect_ratio: Optional[float] = None,
            *,
            reaction_1            
            ):


        self.T = self.T_default if T is None else T
        self.P = self.P_default if P is None else P
        self.tau = self.tau_default if tau is None else tau
        self.vessel_material = 'Stainless steel 316' if vessel_material is None else vessel_material
        self.vessel_type = 'Vertical' if vessel_type is None else vessel_type
        self.tau_0 = self.tau_0_default if tau_0 is None else tau_0
        self.catalyst_bulk_density = self.catalyst_bulk_density if catalyst_bulk_density is None else catalyst_bulk_density
        self.free_frac      = self.free_frac_default      if free_frac      is None else free_frac
        self.V_max = self.V_max_default if V_max is None else V_max
        self.aspect_ratio          = self.aspect_ratio         if aspect_ratio          is None else aspect_ratio
        self.reaction = reaction_1
        # heat_exchanger_1 = self.auxiliary('heat_exchanger_1', bst.HXutility, pump_1.outs[0])






    def _size_bed(self):


        cycle_time        = self.tau + self.tau_0                  


        # Total monomer flow
        total_monomer_flow = self.ins[0].F_vol                      # [m3/hr]

        V_theoretical = total_monomer_flow * self.tau               # [m3] Theoretical volume required
        
        V_actual = V_theoretical/(1+self.free_frac)                 # [m3] Actual volume required based on free fraction
        
        N_working = ceil(V_actual/self.V_max)                       # Number of working reactors based off maximum volume
        N_offline = ceil(N_working*(self.tau_0/cycle_time))         # Number of offline beds, calculated based off cleaning time and the total cycle time, rounded off to the next number
        N_total = N_working + N_offline                             # Total beds required
        V_total = N_total * self.V_max                              # Total volume required
        
        diameter = (4*self.V_max)/((self.aspect_ratio*np.pi)**1/3)
        length = self.aspect_ratio * diameter


        return length, diameter, N_total, N_working, V_total

        
    def _run(self):
        inf, = self.ins
        eff, = self.outs

        eff.mix_from(self.ins)
        self.reaction(eff)


        eff.P = self.P                                             # Assuming no P drop
        eff.T = self.T                                             # Assuming isothermal operation
    



        

    def _design(self):
        length, diameter, N_total, N_working, V_total = self._size_bed()   # Calling size bed function to determine diameter and length 
        
        cycle_time = self.tau + self.tau_0
        self.set_design_result('Diameter', 'ft', diameter)
        self.set_design_result('Length', 'ft', length)
        self.set_design_result('Reactor volume', 'm3', self.V_max)
        self.set_design_result('Total volume', 'm3', V_total)
        self.set_design_result('Total beds', '', N_total)
        self.set_design_result('Beds in service', '', N_working)
        self.set_design_result('Time on stream', 'hr', self.tau)
        self.set_design_result('Turnaround time', 'hr', self.tau_0)
        self.set_design_result('Batch time', 'hr', cycle_time)


        
        
        # Calculates weight based off pressure, diameter and length
        # Adds vcessel type wall thickness, vessel weight, diameter and length to dictionary
        # But diameter and length are already there because of set_design_result above
        
        self.design_results.update(
            self._vertical_vessel_design(    
                self.P*(1/6894.76),
                self.design_results['Diameter']*3.28084,
                self.design_results['Length']*3.28084
            )
        )
        
                            

        #duty = -70 * self.ins[0].imol['Hydrogen']   # Aromatic hydrogenation has duty between 58-70 kJ/mol H2 according to https://www.nature.com/articles/s41563-024-02024-6

        #self.add_heat_utility(duty/N_total, self.T)
        #self.set_design_result('Duty', 'kJ/hr', duty)





    def _cost(self):
        design = self.design_results # Calling the dictionary used to store design results in design method above 

        baseline_purchase_costs = self.baseline_purchase_costs # Dictionary for storing baseline costs

        weight = design['Weight']  # weight parameter stores the value from the 'Weight' key in the design dictionnary
        
        N_reactors = design['Total beds']
        # Calculates the baseline purchase cost based off diameter length and weight
        baseline_purchase_costs.update( 
            self._vessel_purchase_cost(weight, design['Diameter'], design['Length'])
        )

        self.parallel['self'] = N_reactors # Used to create multiple of the same beds


        
       
        """
        ---------
          
        Parameters that can be further fine-tuned based on industry/national lab data
        - Void fraction of poplar bed: Herein assumed 0.5, this is subject to change
        - Working volue fraction: Herein assumed 80%, but can change depending on how well mass transfer occurs in real reactors
        - V_max: Maximum volume of a single reactor, herein assumed as 600 m3 based on Bartling et al 2021 paper, but subject to change
        - residence time: Herein 2 hrs, but could change based on which regime is more limiting. 


        ----------

        """

    

In [15]:
HDO = HydrodeoxygenationReactor(ins = compressor-0, reaction_1 = hydrodeoxygenation_rxn)
HDO.simulate()

C:\Users\hwadg\AppData\Local\Temp\ipykernel_18548\1034142330.py:180: CostWarning: <HydrodeoxygenationReactor: R1> Vertical vessel weight (1.019e+12 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  self._vertical_vessel_design(
C:\Users\hwadg\AppData\Local\Temp\ipykernel_18548\1034142330.py:180: CostWarning: <HydrodeoxygenationReactor: R1> Vertical vessel length (7519 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(


In [16]:
HDO.results()

Hydrodeoxygenation reactor                          Units        R1
Design              Diameter                           ft   1.5e+03
                    Length                             ft  7.52e+03
                    Reactor volume                     m3       600
                    Total volume                       m3      1200
                    Total beds                                    2
                    Beds in service                               1
                    Time on stream                     hr         2
                    Turnaround time                    hr         1
                    Batch time                         hr         3
                    Vessel type                            Vertical
                    Weight                             lb  1.02e+12
                    Wall thickness                     in       574
Purchase cost       Vertical pressure vessel (x2)     USD  3.33e+13
                    Platform and ladders (x2)         USD  9.63e+07
Total purchase cost                                   USD  3.33e+13
Utility cost                                       USD/hr         0

In [17]:
HDO

HydrodeoxygenationReactor: R1
ins...
[0] s45  from  IsentropicCompressor-K1
    phases: ('g', 'l'), T: 556.03 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen        300
                    (l) Propylguaiacol  17.4
                        Propylsyringol  14.7
outs...
[0] s46  
    phases: ('g', 'l'), T: 573.15 K, P: 5e+06 Pa
    flow (kmol/hr): (g) Hydrogen           77.7
                        Methane            46.9
                    (l) Water              79
                        propylcyclohexane  32.1
